In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langchain_core.messages import SystemMessage, HumanMessage
from typing import Literal

load_dotenv()

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0
)

In [ ]:
loader = PyPDFLoader(
    "/mnt/Scratch/codebase/rag-projects/optirag/data/Stock_Market_Performance_2024.pdf"
)

documents = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120
)

chunks = splitter.split_documents(documents)

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [ ]:
from typing import TypedDict

class RAGState(TypedDict):
    question: str
    documents: list
    generation: str
    retries: int

In [ ]:
def retrieve(state: RAGState):
    docs = retriever.invoke(state["question"])

    return {
        "documents": docs,
        "retries": state.get("retries", 0)
    }

In [ ]:
def grade_documents(state: RAGState) -> Literal["generate", "rewrite"]:
    question = state["question"]
    docs = state["documents"]

    if not docs:
        return "rewrite"

    content = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
    Question: {question}
    Documents: {content}

    Are these documents relevant to answering the question?
    Return only YES or NO.
    """

    result = llm.invoke(prompt).content.strip().upper()

    if result.startswith("YES"):
        return "generate"

    return "rewrite"